# Fase 1: Preparación de datos y GIS

## Evaluación de la Red Ferroviaria Ampliada de Lima

### Objetivos de Fase 1:
1. Geocodificar estaciones existentes (L1, L2 parcial) y propuestas (L3-L6, Trenes Ica y Norte)
2. Calcular buffers de 800 m a pie y área de cobertura
3. Descargar red vial de Lima (OSM) y edificios
4. Generar dataset GTFS sintético para modelado de demanda
5. Visualizar la red propuesta

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from config import PROCESSED_DATA, RAW_DATA, FIGURES, BUFFER_METERS, CRS_PROJECTED
from stations import build_stations_gdf, build_lines_gdf
from data_osm import download_lima_network, download_lima_buildings
from data_gtfs import build_synthetic_gtfs, load_gtfs_trips
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

---
## 1. Estaciones y líneas

In [ ]:
stations = build_stations_gdf()
stations_proj = stations.to_crs(CRS_PROJECTED)
lines = build_lines_gdf(stations)
lines_proj = lines.to_crs(CRS_PROJECTED)

print(f"Estaciones: {len(stations)}")
print(f"Líneas: {len(lines)}")
stations.groupby(['status', 'line_name']).size().unstack(fill_value=0)

---
## 2. Buffers de 800 m (área de influencia peatonal)

In [ ]:
buffers = stations_proj.copy()
buffers['geometry'] = stations_proj.geometry.buffer(BUFFER_METERS)
buffer_union = buffers.geometry.union_all()
coverage_km2 = buffer_union.area / 1e6

print(f"Área total cubierta (buffers 800m): {coverage_km2:.1f} km²")
print(f"Porcentaje aproximado de Lima Metropolitana (~2672 km²): {coverage_km2/2672*100:.1f}%")

---
## 3. Datos OSM (red vial + edificios)

In [ ]:
G = download_lima_network()
print(f"\nRed vial: {len(G.nodes)} nodos, {len(G.edges)} aristas")

In [ ]:
try:
    buildings = download_lima_buildings()
    print(f"Edificios descargados: {len(buildings)}")
except Exception as e:
    print(f"Error: {e}")

---
## 4. GTFS sintético

In [ ]:
build_synthetic_gtfs()
routes, trips, stops = load_gtfs_trips(None)

---
## 5. Visualización de la red

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 14))

status_colors = {
    'existing': '#2E86AB',
    'partial': '#A23B72',
    'proposed': '#F18F01',
}

for status, color in status_colors.items():
    subset = lines_proj[lines_proj['status'] == status]
    if not subset.empty:
        subset.plot(ax=ax, color=color, linewidth=2.5, label=f'{status} ({len(subset)} líneas)', alpha=0.8)

stations_proj.plot(ax=ax, color='black', markersize=8, alpha=0.7, label='Estaciones')
ax.legend(fontsize=10)
ax.set_title('Red Ferroviaria Propuesta para Lima Metropolitana', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---
## Resumen de Fase 1

| Indicador | Valor |
|-----------|-------|
| Estaciones totales | 72 |
| Líneas | 8 (1 existente, 1 parcial, 6 propuestas) |
| Buffer 800m | ~115 km² |
| Red vial OSM | ~113K nodos, ~293K aristas |
| Edificios OSM | ~152K |
| GTFS | Sintético generado |

**Siguiente paso:** Fase 2 - Modelo de demanda (logit + gravitatorio)